In [32]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

## Prerequisites

In [33]:
import pandas as pd
import pickle

from radp.digital_twin.utils.gis_tools import GISTools
from notebooks.radp_library import calculate_received_power, get_ues_cells_cartesian_df, calc_rx_power, calc_log_distance, calc_relative_bearing, preprocess_ue_data
from notebooks.radp_library import get_percell_data

# Curating training/update data

training/update data: key value pairs of cell_id vs processed df (have engineered features). format:

```bash
{
    "cell_1": df1,
    "cell_2": df2,
    "cell_3": df3,
    ...      
}
```

In [34]:
# place the pkl file in the same directory as this script, downloadable from maveric drive

training_pickle_path = Path('notebooks/data/sim_data/processed_training_data.pkl')
absolute_pickle_path =Path().absolute().parent / training_pickle_path
with open(absolute_pickle_path, 'rb') as f:
    training_data = pickle.load(f)

print(training_data)

{'cell_1':       mock_ue_id   longitude   latitude  tick  cell_lat  cell_lon cell_id  \
0              5  -43.264737 -30.862211    20     -90.0    -180.0  cell_1   
1             10   54.978193 -55.259842    59     -90.0    -180.0  cell_1   
2             12  -98.676642  40.779019    56     -90.0    -180.0  cell_1   
3             11    5.606533  12.413489    36     -90.0    -180.0  cell_1   
4             14 -134.815655 -58.151045    87     -90.0    -180.0  cell_1   
...          ...         ...        ...   ...       ...       ...     ...   
1995          15   86.855787  41.319434    41     -90.0    -180.0  cell_1   
1996          16  -84.432536 -38.512045    60     -90.0    -180.0  cell_1   
1997          13  142.954432 -16.862212    82     -90.0    -180.0  cell_1   
1998          19   89.254786 -82.596233    27     -90.0    -180.0  cell_1   
1999           4   20.088340 -55.786882    34     -90.0    -180.0  cell_1   

      cell_az_deg  cell_carrier_freq_mhz  log_distance  cell_rxp

# loading ue and topology

In [35]:
# place the required files in the same directory as this script, downloadable from maveric drive

simple_ue = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

In [36]:
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [37]:
simple_ue = simple_ue.loc[:, ['longitude', 'latitude']]
simple_ue.head()

,longitude,latitude
0,-22.625309,59.806764
1,119.764151,54.857584
2,72.095437,-20.253892
3,-67.548009,-38.100941
4,59.867089,-83.103930


# Functions

## Preprocess functions

check inside `radp_library.py`:
1. get_ues_cells_cartesian_df
2. calc_log_distance
3. calcalculate_received_power
4. calc_relative_bearing
5. preprocess_ue_data (calls above 1-3)

**used _ in function names to separate from radp library import**

In [38]:
# f0

def _get_ues_cells_cartesian_df(data,topology):
    """returns a cartesian dataframe of UE and cell data"""
    if topology["cell_id"].dtype == object:
            topology["cell_id"] = (
                topology["cell_id"].str.replace("cell_", "").astype(int)
            )
    data["key"] = 1
    topology["key"] = 1
    cartesian_df = pd.merge(data, topology, on="key").drop("key", axis=1)
    
    data.drop(columns=["key"], inplace=True)
    topology.drop(columns=["key"], inplace=True)
    return cartesian_df

In [39]:
# f1

def _calc_log_distance(cartesian_df):
    """adds a log distance column to the cartesian dataframe based on the lat/lon of the UE and cell"""
    cartesian_df["log_distance"] = cartesian_df.apply(
        lambda row: GISTools.get_log_distance(
            row["latitude"], row["longitude"], row["cell_lat"], row["cell_lon"]
        ),
        axis=1,
    )
    return cartesian_df

In [40]:
# f2

def _calc_rx_power(cartesian_df):
    """adds a cell_rxpwr_dbm column to the cartesian dataframe based on the log distance and cell frequency using fspl"""
    cartesian_df["cell_rxpwr_dbm"] = cartesian_df.apply(
        lambda row: calculate_received_power(
            row["log_distance"], row["cell_carrier_freq_mhz"]
        ),
        axis=1,
    )
    return cartesian_df

In [41]:
# f3

def _calc_relative_bearing(cartesian_df):
    """adds a relative_bearing column to the cartesian dataframe based on the lat/lon of the UE and cell and az_deg of the cell"""
    cartesian_df["relative_bearing"] = cartesian_df.apply(
        lambda row: GISTools.get_relative_bearing(
            row["cell_az_deg"],
            row["cell_lat"],
            row["cell_lon"],
            row["latitude"],
            row["longitude"],
        ),
        axis=1,
    )
    return cartesian_df

In [42]:
# f4

def _preprocess_ue_data(data, topology):
    """creates a cartesian dataframe of UE and cell data, adds log distance and rx power columns"""
    cartesian_df = get_ues_cells_cartesian_df(data, topology)
    cartesian_df = calc_log_distance(cartesian_df)
    return calc_rx_power(cartesian_df)


## Prepare train or update data function

prepares key value pair data format for training or updating

In [43]:
def prepare_train_or_update_data(df):
    update_data = calc_log_distance(df)
    update_data = calc_relative_bearing(update_data)
    update_data.drop(columns=['longitude', 'latitude','cell_lat','cell_lon', 'cell_az_deg','cell_carrier_freq_mhz'], inplace=True)

    train_per_cell_df = [x for _, x in update_data.groupby("cell_id")]
    n_cell = len(topology.index)

    metadata_df = pd.DataFrame(
        {
            "cell_id": [cell_id for cell_id in topology.cell_id],
            "idx": [i + 1 for i in range(n_cell)],
        }
    )
    
    idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
    n_samples_train = []
    
    for df in train_per_cell_df:
        n_samples_train.append(df.shape[0])

    train_per_cell_df_processed = []
    for i in range(n_cell):
        train_per_cell_df_processed.append(
            get_percell_data(
                data_in=train_per_cell_df[i],
                choose_strongest_samples_percell=False,
                n_samples=n_samples_train[i],
            )[0][0]
        )

    training_data = {}

    for i, df in enumerate(train_per_cell_df_processed):
        train_cell_id = idx_cell_id_mapping[i + 1]
        training_data[train_cell_id] = df
    
    return training_data

## Bebugging

classic pranto moment :p

In [44]:
simple_ue.head()

,longitude,latitude
0,-22.625309,59.806764
1,119.764151,54.857584
2,72.095437,-20.253892
3,-67.548009,-38.100941
4,59.867089,-83.103930


In [45]:
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [46]:
cartesian_df = _get_ues_cells_cartesian_df(simple_ue, topology)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100
1,-22.625309,59.806764,0.0,0.0,2,120,2100
2,-22.625309,59.806764,90.0,180.0,3,240,2100
3,119.764151,54.857584,-90.0,-180.0,1,0,2100
4,119.764151,54.857584,0.0,0.0,2,120,2100
...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100
5998,34.222834,63.970820,0.0,0.0,2,120,2100


In [47]:
cartesian_df = _calc_log_distance(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100,16.629500
1,-22.625309,59.806764,0.0,0.0,2,120,2100,15.752768
2,-22.625309,59.806764,90.0,180.0,3,240,2100,15.027772
3,119.764151,54.857584,-90.0,-180.0,1,0,2100,16.595905
4,119.764151,54.857584,0.0,0.0,2,120,2100,16.289273
...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100,15.054748
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100,16.379387
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100,16.656917
5998,34.222834,63.970820,0.0,0.0,2,120,2100,15.850264


In [48]:
cartesian_df = _calc_rx_power(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100,16.629500,-100.311970
1,-22.625309,59.806764,0.0,0.0,2,120,2100,15.752768,-99.841523
2,-22.625309,59.806764,90.0,180.0,3,240,2100,15.027772,-99.432278
3,119.764151,54.857584,-90.0,-180.0,1,0,2100,16.595905,-100.294405
4,119.764151,54.857584,0.0,0.0,2,120,2100,16.289273,-100.132420
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100,15.054748,-99.447856
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100,16.379387,-100.180339
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100,16.656917,-100.326278
5998,34.222834,63.970820,0.0,0.0,2,120,2100,15.850264,-99.895116


In [49]:
# this call should be same as previous df but with different function call

cartesian_df = _preprocess_ue_data(simple_ue, topology)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100,16.629500,-100.311970
1,-22.625309,59.806764,0.0,0.0,2,120,2100,15.752768,-99.841523
2,-22.625309,59.806764,90.0,180.0,3,240,2100,15.027772,-99.432278
3,119.764151,54.857584,-90.0,-180.0,1,0,2100,16.595905,-100.294405
4,119.764151,54.857584,0.0,0.0,2,120,2100,16.289273,-100.132420
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100,15.054748,-99.447856
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100,16.379387,-100.180339
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100,16.656917,-100.326278
5998,34.222834,63.970820,0.0,0.0,2,120,2100,15.850264,-99.895116


In [50]:
cartesian_df = _calc_relative_bearing(cartesian_df)
cartesian_df

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm,relative_bearing
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100,16.629500,-100.311970,157.374691
1,-22.625309,59.806764,0.0,0.0,2,120,2100,15.752768,-99.841523,227.382797
2,-22.625309,59.806764,90.0,180.0,3,240,2100,15.027772,-99.432278,142.625309
3,119.764151,54.857584,-90.0,-180.0,1,0,2100,16.595905,-100.294405,299.764151
4,119.764151,54.857584,0.0,0.0,2,120,2100,16.289273,-100.132420,271.427214
...,...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100,15.054748,-99.447856,89.471165
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100,16.379387,-100.180339,136.480045
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100,16.656917,-100.326278,214.222834
5998,34.222834,63.970820,0.0,0.0,2,120,2100,15.850264,-99.895116,255.358234


## Rewriting Update (Training From Scratch)

In [51]:
simple_ue = pd.read_csv('data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('data/sim_data/topology.csv')

In [52]:
simple_ue = simple_ue.loc[:, ['longitude', 'latitude']]

simple_ue.head()

,longitude,latitude
0,-22.625309,59.806764
1,119.764151,54.857584
2,72.095437,-20.253892
3,-67.548009,-38.100941
4,59.867089,-83.103930


In [53]:
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,35.690556,139.691944,cell_1,0,2100
1,35.690556,139.691944,cell_2,120,2100
2,35.690556,139.691944,cell_3,240,2100


In [54]:
# assume user/client/developer has the UE data withour rx power information. should call the following function to get the rx power information

df = preprocess_ue_data(simple_ue, topology) # returns the rx power information in cartesian format
df1 = df.copy()

In [55]:
df.head()

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,35.690556,139.691944,1,0,2100,16.043691,-100.000472
1,-22.625309,59.806764,35.690556,139.691944,2,120,2100,16.043691,-100.000472
2,-22.625309,59.806764,35.690556,139.691944,3,240,2100,16.043691,-100.000472
3,119.764151,54.857584,35.690556,139.691944,1,0,2100,14.780124,-99.287948
4,119.764151,54.857584,35.690556,139.691944,2,120,2100,14.780124,-99.287948


In [56]:
bayesian_digital_twins = {}

In [57]:
try:
    if not isinstance(df, pd.DataFrame):
        raise TypeError("The input 'new_data' must be a pandas DataFrame.")

    expected_columns = {"longitude", "latitude", "cell_lat", "cell_lon", "cell_id", "cell_az_deg", "cell_carrier_freq_mhz", "cell_rxpwr_dbm"}
    if not expected_columns.issubset(df.columns):
        raise ValueError(
            f"The input DataFrame must contain the following columns: {expected_columns}"
        )
    
    # ? do we need str cell_id or int cell_id? does both work?
    df["cell_id"] = df["cell_id"].apply(lambda x: f"cell_{x}")
    topology["cell_id"] = topology["cell_id"].apply(lambda x: f"cell_{x}")
    print(df)
    
    prepared_data = prepare_train_or_update_data(df)
    print(prepared_data)
    if bayesian_digital_twins:
        # TODO: Update BDT
        pass
    else:
        # TODO: Create BDT from scratch
        pass

        # TODO: Add Train

except TypeError as te:
    print(f"TypeError: {te}")
except ValueError as ve:
    print(f"ValueError: {ve}")
except KeyError as ke:
    print(f"KeyError: {ke}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

       longitude   latitude   cell_lat    cell_lon cell_id  cell_az_deg  \
0     -22.625309  59.806764  35.690556  139.691944  cell_1            0   
1     -22.625309  59.806764  35.690556  139.691944  cell_2          120   
2     -22.625309  59.806764  35.690556  139.691944  cell_3          240   
3     119.764151  54.857584  35.690556  139.691944  cell_1            0   
4     119.764151  54.857584  35.690556  139.691944  cell_2          120   
...          ...        ...        ...         ...     ...          ...   
5995  -16.480045 -26.656397  35.690556  139.691944  cell_2          120   
5996  -16.480045 -26.656397  35.690556  139.691944  cell_3          240   
5997   34.222834  63.970820  35.690556  139.691944  cell_1            0   
5998   34.222834  63.970820  35.690556  139.691944  cell_2          120   
5999   34.222834  63.970820  35.690556  139.691944  cell_3          240   

      cell_carrier_freq_mhz  log_distance  cell_rxpwr_dbm  
0                      2100     16.0436

### Testing

In [58]:
update_data = calc_log_distance(df1)
update_data = calc_relative_bearing(update_data)

update_data.drop(columns=['longitude', 'latitude','cell_lat','cell_lon', 'cell_az_deg','cell_carrier_freq_mhz'], inplace=True)

In [59]:
update_data

,cell_id,log_distance,cell_rxpwr_dbm,relative_bearing
0,1,16.043691,-100.000472,351.153872
1,2,16.043691,-100.000472,231.153872
2,3,16.043691,-100.000472,111.153872
3,1,14.780124,-99.287948,330.617727
4,2,14.780124,-99.287948,210.617727
...,...,...,...,...
5995,2,16.681342,-100.339006,167.318054
5996,3,16.681342,-100.339006,47.318054
5997,1,15.788136,-99.861003,332.079388
5998,2,15.788136,-99.861003,212.079388


In [60]:
train_per_cell_df = [x for _, x in update_data.groupby("cell_id")]
n_cell = len(topology.index)

metadata_df = pd.DataFrame(
    {
        "cell_id": [cell_id for cell_id in topology.cell_id],
        "idx": [i + 1 for i in range(n_cell)],
    }
)
idx_cell_id_mapping = dict(zip(metadata_df.idx, metadata_df.cell_id))
desired_idxs = [1 + r for r in range(n_cell)]

n_samples_train = []
for df in train_per_cell_df:
    n_samples_train.append(df.shape[0])

train_per_cell_df_processed = []
for i in range(n_cell):
    train_per_cell_df_processed.append(
        get_percell_data(
            data_in=train_per_cell_df[i],
            choose_strongest_samples_percell=False,
            n_samples=n_samples_train[i],
        )[0][0]
    )

training_data = {}

for i, df in enumerate(train_per_cell_df_processed):
    train_cell_id = idx_cell_id_mapping[i + 1]
    training_data[train_cell_id] = df

In [61]:
training_data

{'cell_1':       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
 0           1     16.782517     -100.391528         27.934319
 1           1     16.372865     -100.176879        219.107651
 2           1     16.081985     -100.021179         40.229093
 3           1     16.367184     -100.173865        309.143895
 4           1     16.386669     -100.184200        143.622155
 ...       ...           ...             ...               ...
 1995        1     15.335904      -99.608574        294.404550
 1996        1     16.597103     -100.295032        108.097728
 1997        1     15.583699      -99.747797        176.071117
 1998        1     16.414202     -100.198782        186.640616
 1999        1     16.525990     -100.257736        223.811745
 
 [2000 rows x 4 columns],
 'cell_2':       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
 0           2     16.782517     -100.391528        267.934319
 1           2     16.372865     -100.176879         99.107651
 2     

In [62]:
print(training_data)

{'cell_1':       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
0           1     16.782517     -100.391528         27.934319
1           1     16.372865     -100.176879        219.107651
2           1     16.081985     -100.021179         40.229093
3           1     16.367184     -100.173865        309.143895
4           1     16.386669     -100.184200        143.622155
...       ...           ...             ...               ...
1995        1     15.335904      -99.608574        294.404550
1996        1     16.597103     -100.295032        108.097728
1997        1     15.583699      -99.747797        176.071117
1998        1     16.414202     -100.198782        186.640616
1999        1     16.525990     -100.257736        223.811745

[2000 rows x 4 columns], 'cell_2':       cell_id  log_distance  cell_rxpwr_dbm  relative_bearing
0           2     16.782517     -100.391528        267.934319
1           2     16.372865     -100.176879         99.107651
2           2     16.08